# EPL Home Advantage Analysis

Five Premier League seasons (2021-22 to 2025-26, 1,900 matches) from [football-data.co.uk](https://www.football-data.co.uk).

Run this notebook from the `Scripts/` folder: it reads the season files in `../Data/` and writes `../Data/epl_5seasons_combined.csv` (all matches in one table) and `../Data/ppg_by_team_season.csv`, the data source for the [Tableau Public dashboard](https://public.tableau.com/app/profile/riyan.qureshi/viz/EPL_HOME_ADVANTAGE_ANALYSIS/Dashboard1).

In [1]:
import glob
import sqlite3

import numpy as np
import pandas as pd
from scipy import stats

## 1. Load and combine five seasons

The season files don't share the same layout (between 106 and 132 columns), so only the 21 fields used in the analysis are kept.

In [2]:
cols = [
    'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR', 'HS', 'AS', 'HST', 'AST',
    'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]

files = sorted(glob.glob('../Data/E0_*.csv'))

dfs = []
for f in files:
    season = f.split('E0_')[1].replace('.csv', '')
    raw = pd.read_csv(f)
    print(f"{season}: {raw.shape[1]} columns in the source file")
    df = raw[cols].copy()
    df['Season'] = season
    dfs.append(df)

all_seasons = pd.concat(dfs, ignore_index=True)

print(f"\nLoaded {len(all_seasons)} matches")
print(f"Seasons: {all_seasons['Season'].nunique()}")

all_seasons.to_csv('../Data/epl_5seasons_combined.csv', index=False)
print("Saved the combined table to ../Data/epl_5seasons_combined.csv")

2021-2022: 106 columns in the source file
2022-2023: 106 columns in the source file
2023-2024: 106 columns in the source file
2024-2025: 120 columns in the source file
2025-2026: 132 columns in the source file

Loaded 1900 matches
Seasons: 5
Saved the combined table to ../Data/epl_5seasons_combined.csv


## 2. Home vs away points per game (PPG)

Each result becomes points (win = 3, draw = 1, loss = 0), averaged per team, per season, split into home and away games.

In [3]:
all_seasons['HomePoints'] = all_seasons['FTR'].map({'H': 3, 'D': 1, 'A': 0})
all_seasons['AwayPoints'] = all_seasons['FTR'].map({'A': 3, 'D': 1, 'H': 0})

home = (
    all_seasons
    .groupby(['Season', 'HomeTeam'])['HomePoints']
    .mean()
    .reset_index()
    .rename(columns={'HomeTeam': 'Team', 'HomePoints': 'HomePPG'})
)

away = (
    all_seasons
    .groupby(['Season', 'AwayTeam'])['AwayPoints']
    .mean()
    .reset_index()
    .rename(columns={'AwayTeam': 'Team', 'AwayPoints': 'AwayPPG'})
)

ppg = home.merge(away, on=['Season', 'Team'])
ppg['PPG_Diff'] = ppg['HomePPG'] - ppg['AwayPPG']

print(f"Team-seasons: {len(ppg)}")

Team-seasons: 100


## 3. Is the home advantage statistically significant?

A paired t-test compares each team's home PPG with its own away PPG, across all 100 team-seasons.

In [4]:
print("Average PPG:")
print(ppg[['HomePPG', 'AwayPPG']].mean())

t_stat, p_value = stats.ttest_rel(ppg['HomePPG'], ppg['AwayPPG'])

print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.1e}" + ("  (p < 0.001)" if p_value < 0.001 else ""))

# Each team plays 19 home games a season, so the per-game gap x 19 is the seasonal value
extra_pts_per_season = (ppg['HomePPG'].mean() - ppg['AwayPPG'].mean()) * 19
print(f"Estimated home advantage: ~{extra_pts_per_season:.1f} points per season")

Average PPG:
HomePPG    1.563684
AwayPPG    1.197368
dtype: float64

t-statistic: 9.741
p-value: 4.0e-16  (p < 0.001)
Estimated home advantage: ~7.0 points per season


## 4. Season-by-season trend and team ranking

In [5]:
season_gap = ppg.groupby('Season')['PPG_Diff'].mean()
print("Home advantage (PPG) by season:")
print(season_gap.round(4))

# Straight-line trend across the five seasons (least-squares fit)
x = np.arange(len(season_gap))
slope, intercept = np.polyfit(x, season_gap.values, 1)
print(f"\nTrend line: {intercept:.2f} in {season_gap.index[0]} -> "
      f"{intercept + slope * x[-1]:.2f} in {season_gap.index[-1]} ({slope:+.3f} PPG per season)")

team = (
    ppg.groupby('Team')
    .agg(PPG_Diff=('PPG_Diff', 'mean'), Seasons=('Season', 'count'))
    .sort_values('PPG_Diff', ascending=False)
)
print("\nStrongest home advantage:")
print(team.head(5).round(3))
print("\nWeakest home advantage:")
print(team.tail(5).round(3))

Home advantage (PPG) by season:
Season
2021-2022    0.2684
2022-2023    0.5921
2023-2024    0.4105
2024-2025    0.1816
2025-2026    0.3789
Name: PPG_Diff, dtype: float64

Trend line: 0.40 in 2021-2022 -> 0.33 in 2025-2026 (-0.019 PPG per season)

Strongest home advantage:
             PPG_Diff  Seasons
Team                          
Newcastle       0.663        5
Liverpool       0.642        5
Sunderland      0.632        1
Man United      0.611        5
Aston Villa     0.558        5

Weakest home advantage:
             PPG_Diff  Seasons
Team                          
Burnley         0.123        3
Southampton     0.123        3
Norwich         0.105        1
Watford        -0.368        1
Ipswich        -0.421        1


## 5. Cross-check the calculation in SQL (SQLite)

The same PPG calculation, rebuilt as a SQL query over the match data, must return identical numbers. An in-memory database is used, so nothing is left on disk.

In [6]:
conn = sqlite3.connect(':memory:')
all_seasons.to_sql('matches', conn, if_exists='replace', index=False)

query = """
WITH home_ppg AS (
    SELECT
        Season,
        HomeTeam AS Team,
        AVG(
            CASE
                WHEN FTR = 'H' THEN 3
                WHEN FTR = 'D' THEN 1
                ELSE 0
            END
        ) AS HomePPG
    FROM matches
    GROUP BY Season, HomeTeam
),

away_ppg AS (
    SELECT
        Season,
        AwayTeam AS Team,
        AVG(
            CASE
                WHEN FTR = 'A' THEN 3
                WHEN FTR = 'D' THEN 1
                ELSE 0
            END
        ) AS AwayPPG
    FROM matches
    GROUP BY Season, AwayTeam
)

SELECT
    h.Season,
    h.Team,
    h.HomePPG,
    a.AwayPPG,
    h.HomePPG - a.AwayPPG AS PPG_Diff

FROM home_ppg h

JOIN away_ppg a
    ON h.Season = a.Season
    AND h.Team = a.Team

ORDER BY h.Season, h.Team;
"""

result = pd.read_sql(query, conn)
conn.close()

check = ppg.merge(result, on=['Season', 'Team'], suffixes=('_pandas', '_sql'))
max_diff = max(
    (check['HomePPG_pandas'] - check['HomePPG_sql']).abs().max(),
    (check['AwayPPG_pandas'] - check['AwayPPG_sql']).abs().max(),
)
print(f"Team-seasons compared: {len(check)} of {len(ppg)}")
print(f"Largest difference between Pandas and SQL: {max_diff:.1e}")

Team-seasons compared: 100 of 100
Largest difference between Pandas and SQL: 0.0e+00


## 6. Save the output used by the Tableau dashboard

In [7]:
result.to_csv('../Data/ppg_by_team_season.csv', index=False)
print(f"Wrote {len(result)} rows to ../Data/ppg_by_team_season.csv")

Wrote 100 rows to ../Data/ppg_by_team_season.csv
